# Late Attestations timings v2 - Quick models

#### Maria Silva, September 2025

## 1. Imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Main directories and files
current_path = os.getcwd()
repo_dir = os.path.abspath(os.path.join(current_path, ".."))
data_dir = os.path.join(repo_dir, "data")

# Add src to path
sys.path.append(os.path.join(repo_dir, "src"))

In [3]:
from late_atts_model_runner import load_data

In [4]:
df = load_data(data_dir, "sample_12243620_12294020")
df = df[df["header_from"]!="self_build"]
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 5263471 entries, 1060 to 10564738
Data columns (total 20 columns):
 #   Column                        Non-Null Count    Dtype         
---  ------                        --------------    -----         
 0   slot                          5263471 non-null  int64         
 1   atts_validator                5263471 non-null  int64         
 2   atts_subnet                   5263471 non-null  object        
 3   atts_arrival_time_ms          5263471 non-null  int64         
 4   request_datetime              4688856 non-null  datetime64[ns]
 5   publish_datetime              4688856 non-null  datetime64[ns]
 6   header_from                   5263471 non-null  object        
 7   publish_time_ms               5263471 non-null  float64       
 8   slot_start_datetime           5263471 non-null  datetime64[ns]
 9   request_time_ms               4688856 non-null  float64       
 10  p95_block_arrival_ms          5263471 non-null  int64         
 11 

## 2. Linear regression model

In [5]:
features = [
    "header_from",
    "p66_block_arrival_ms",
    "block_total_bytes_compressed",
    "block_gas_used",
    "block_blob_count",
    "block_tx_count",
]
cat_features = ["header_from"]
predictor = "net_atts_arrival_time_ms"

In [6]:
# Reduce entity dimensions
entity_counts = df["entity"].value_counts() / len(df)
model_df = df.merge(entity_counts, left_on="entity", right_index=True)
model_df["entity"] = np.where(model_df["count"] > 0.01, model_df["entity"], "other")
# Filter rows with negative arrival times
model_df = model_df[model_df[predictor] >= 0]
# Select features
X_raw = model_df[features].values
# Build the column transformer
cat_indices = [features.index(x) for x in features if x in cat_features]
num_indices = [features.index(x) for x in features if x not in cat_features]
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_indices),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_indices),
    ],
    sparse_threshold=0,
)
# Fit and transform X
X = preprocessor.fit_transform(X_raw)
# Define the target: binary classification: late (1) or not (0)
y = model_df[predictor].values

In [7]:
num_feature_names = np.array([x for x in features if x not in cat_features])
cat_feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(cat_features)
feature_names = np.concatenate([num_feature_names, cat_feature_names])

In [8]:
X_df = pd.DataFrame(X, columns=feature_names)
X_with_intercept_df = sm.add_constant(X_df)  # adds intercept

logit_model = sm.OLS(y, X_with_intercept_df)
result = logit_model.fit()
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.052
Model:                            OLS   Adj. R-squared:                  0.052
Method:                 Least Squares   F-statistic:                 3.577e+04
Date:                Wed, 03 Sep 2025   Prob (F-statistic):               0.00
Time:                        14:18:38   Log-Likelihood:            -4.3567e+07
No. Observations:             5261784   AIC:                         8.713e+07
Df Residuals:                 5261775   BIC:                         8.713e+07
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
================================================================================================
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const                        -3.205e+13   2.62e+13     -1.223      0.221   -8.34e+13    1.93e+13
p66_block_arrival_ms          -221.5285      0.425   -521.188      0.000    -222.362    -220.695
block_total_bytes_compressed    11.8830      0.665     17.862      0.000      10.579      13.187
block_gas_used                  50.8573      0.599     84.940      0.000      49.684      52.031
block_blob_count                46.1305      0.425    108.607      0.000      45.298      46.963
block_tx_count                  -2.6762      0.612     -4.372      0.000      -3.876      -1.476
header_from_flashbots         3.205e+13   2.62e+13      1.223      0.221   -1.93e+13    8.34e+13
header_from_titan             3.205e+13   2.62e+13      1.223      0.221   -1.93e+13    8.34e+13
header_from_ultrasound        3.205e+13   2.62e+13      1.223      0.221   -1.93e+13    8.34e+13
==============================================================================
Omnibus:                    89287.040   Durbin-Watson:                   1.892
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            95324.242
Skew:                           0.308   Prob(JB):                         0.00
Kurtosis:                       3.234   Cond. No.                     1.94e+14
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 3.32e-22. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

## 3. Logit model with only tx count

In [9]:
# Load data
model_out_dir = os.path.join(data_dir, "model_outputs", "26-08-2025_19:35:20", "4_relay")
# Load X, y, and feature names
X = np.load(os.path.join(model_out_dir, "train_data", "X.npy"))
y = np.load(os.path.join(model_out_dir, "train_data", "y.npy"))
feature_names = np.load(
    os.path.join(model_out_dir, "train_data", "feature_names.npy"), allow_pickle=True
).tolist()

In [10]:
# Select only tx count
X_df = pd.DataFrame(X, columns=feature_names)[["block_tx_count"]]
X_with_intercept_df = sm.add_constant(X_df)  # adds intercept
# Teain model
logit_model = sm.Logit(y, X_with_intercept_df)
result = logit_model.fit()
result.summary()

Optimization terminated successfully.
         Current function value: 0.693074
         Iterations 3


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                      y   No. Observations:               117835
Model:                          Logit   Df Residuals:                   117833
Method:                           MLE   Df Model:                            1
Date:                Wed, 03 Sep 2025   Pseudo R-squ.:               0.0001052
Time:                        14:18:38   Log-Likelihood:                -81668.
converged:                       True   LL-Null:                       -81677.
Covariance Type:            nonrobust   LLR p-value:                 3.381e-05
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.0003      0.006      0.056      0.956      -0.011       0.012
block_tx_count     0.0242      0.006      4.145      0.000       0.013       0.036
==================================================================================
"""